Limpieza de datos

**Objetivo:** revisar la calidad de los cuatro archivos, corregir lo que se puede corregir,
marcar lo que no y generar un Excel listo para el análisis (`datos_limpios/`).

In [1]:
import pandas as pd
import numpy as np
import os

os.makedirs("datos_limpios", exist_ok=True)

## 1. Carga de los archivos

In [2]:
compras = pd.read_excel("purchase_metadata_SAMPLE.xlsx")
materiales = pd.read_csv("material_classification_SAMPLE.csv")
benchmark = pd.read_csv("material_price_benchmark_SAMPLE.csv")
capacidad = pd.read_csv("center_storage_capacity_SAMPLE.csv")

print("compras:", compras.shape)
print("materiales:", materiales.shape)
print("benchmark:", benchmark.shape)
print("capacidad:", capacidad.shape)

compras: (165, 12)
materiales: (25, 9)
benchmark: (60, 8)
capacidad: (3, 3)


In [3]:
compras.head()

,POSampleID,SupplierCode,MaterialCode,Center,OrderDate,BuyingMode,OrderQuantity,UnitPriceUSD,TransportCostUSD,DaysInStorage,BatchNumber,NegotiatorID
0,PO10001-P,V-100070,MAT-1006,P,2024-01-24,Contract,829,26.67,443.79,45.0,B6966,A.Villalba
1,PO10002-P,V-100077,MAT-1007,P,2024-05-05,Spot,1400,24.99,1400.94,116.0,B8776,A.Villalba
2,PO10003-P,V-100049,MAT-1001,P,2024-05-07,Contract,1097,31.99,1275.31,15.0,B3357,L.Fonseca
3,PO10004-P,V-100014,MAT-1008,P,2024-07-20,Spot,382,4.64,138.79,114.0,B3543,L.Fonseca
4,PO10005-S,V-100112,MAT-1005,S,2024-03-17,Spot,1572,6.54,754.74,150.0,B8032,M.Cardenas


In [4]:
compras.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 165 entries, 0 to 164
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   POSampleID        165 non-null    object 
 1   SupplierCode      165 non-null    object 
 2   MaterialCode      165 non-null    object 
 3   Center            165 non-null    object 
 4   OrderDate         165 non-null    object 
 5   BuyingMode        165 non-null    object 
 6   OrderQuantity     165 non-null    int64  
 7   UnitPriceUSD      165 non-null    float64
 8   TransportCostUSD  161 non-null    float64
 9   DaysInStorage     160 non-null    float64
 10  BatchNumber       165 non-null    object 
 11  NegotiatorID      153 non-null    object 
dtypes: float64(3), int64(1), object(8)
memory usage: 15.6+ KB


## 2. Exploración inicial de compras

In [5]:
# Valores nulos por columna
compras.isna().sum()

POSampleID           0
SupplierCode         0
MaterialCode         0
Center               0
OrderDate            0
BuyingMode           0
OrderQuantity        0
UnitPriceUSD         0
TransportCostUSD     4
DaysInStorage        5
BatchNumber          0
NegotiatorID        12
dtype: int64

In [6]:
# ¿Hay órdenes duplicadas?
print("POSampleID duplicados:", compras["POSampleID"].duplicated().sum())

POSampleID duplicados: 0


In [7]:
compras["BuyingMode"].value_counts()

Contract    99
Spot        66
Name: BuyingMode, dtype: int64

In [8]:
compras["Center"].value_counts()

N    57
S    55
P    53
Name: Center, dtype: int64

In [9]:
compras[["OrderQuantity", "UnitPriceUSD", "TransportCostUSD", "DaysInStorage"]].describe()

,OrderQuantity,UnitPriceUSD,TransportCostUSD,DaysInStorage
count,165.000000,165.000000,161.000000,160.000000
mean,1992.000000,19.111697,1707.739068,59.981250
std,5863.775964,10.433320,7420.645809,49.313459
min,277.000000,2.390000,43.160000,3.000000
25%,769.000000,8.490000,340.980000,22.000000
50%,1138.000000,20.050000,746.800000,36.000000
75%,1645.000000,28.040000,1275.310000,101.500000
max,60785.000000,37.530000,92754.580000,180.000000


Lo que se ve a primera vista:
- Hay nulos en `TransportCostUSD` (4), `DaysInStorage` (5) y `NegotiatorID` (12).
- `OrderQuantity` tiene un máximo de 60.785 cuando la mediana es ~1.100: posibles valores atípicos.
- No hay órdenes duplicadas.

## 3. Revisión y corrección de problemas en compras

### 3.1 Códigos de material que no existen en el archivo de materiales

In [10]:
codigos_validos = materiales["MaterialCode"].unique()
codigo_invalido = ~compras["MaterialCode"].isin(codigos_validos)
compras.loc[codigo_invalido, "MaterialCode"].value_counts()

MAT-01015    12
MAT-01004     4
Name: MaterialCode, dtype: int64

`MAT-01004` y `MAT-01015` tienen un cero de más. Corresponden a `MAT-1004` y `MAT-1015`
(los códigos sí existen en el catálogo). **Decisión: corregirlos.**

In [11]:
compras["flag_codigo_corregido"] = codigo_invalido
compras["MaterialCode"] = compras["MaterialCode"].str.replace("MAT-0", "MAT-")

# Comprobación: ya no debería quedar ningún código inválido
print("Códigos inválidos después de corregir:", (~compras["MaterialCode"].isin(codigos_validos)).sum())

Códigos inválidos después de corregir: 0


### 3.2 Centro vs. sufijo del POSampleID
Según el contexto, la letra final del POSampleID es el centro que recibe (N, S, P).

In [12]:
compras["sufijo_po"] = compras["POSampleID"].str[-1]
compras["flag_centro_inconsistente"] = compras["sufijo_po"] != compras["Center"]
compras[compras["flag_centro_inconsistente"]][["POSampleID", "Center", "MaterialCode"]]

,POSampleID,Center,MaterialCode
44,PO10045-S,P,MAT-1011
120,PO10121-N,S,MAT-1014


Dos órdenes no coinciden. No hay forma de saber cuál de los dos campos es el correcto.
**Decisión: dejo la columna `Center`** (el archivo de compras es el registro de lo que se compró) y la marco.

### 3.3 Fechas fuera del alcance (enero - septiembre 2024)

In [13]:
compras["OrderDate"] = pd.to_datetime(compras["OrderDate"])
print("Fecha mínima:", compras["OrderDate"].min())
print("Fecha máxima:", compras["OrderDate"].max())

compras["flag_fuera_de_fecha"] = (compras["OrderDate"] < "2024-01-01") | (compras["OrderDate"] > "2024-09-30")
compras[compras["flag_fuera_de_fecha"]][["POSampleID", "OrderDate", "BuyingMode"]]

Fecha mínima: 2023-12-18 00:00:00
Fecha máxima: 2024-09-27 00:00:00


,POSampleID,OrderDate,BuyingMode
87,PO10088-S,2023-12-18,Contract


Una orden es de diciembre de 2023. **Decisión: excluirla del análisis.**

### 3.4 Valores nulos
- `TransportCostUSD` y `DaysInStorage` son necesarios para calcular el costo total.
  **Decisión: no imputar; esas filas quedan fuera del cálculo de costo** (son pocas: 4 y 5).
- `NegotiatorID` no se usa en el cálculo. **Decisión: reemplazar por "SIN_DATO".**

In [14]:
compras["flag_sin_transporte"] = compras["TransportCostUSD"].isna()
compras["flag_sin_dias"] = compras["DaysInStorage"].isna()
compras["NegotiatorID"] = compras["NegotiatorID"].fillna("SIN_DATO")

compras[compras["flag_sin_transporte"] | compras["flag_sin_dias"]][
    ["POSampleID", "MaterialCode", "BuyingMode", "TransportCostUSD", "DaysInStorage"]]

,POSampleID,MaterialCode,BuyingMode,TransportCostUSD,DaysInStorage
14,PO10015-P,MAT-1009,Contract,NaN,37.0
30,PO10031-P,MAT-1008,Contract,43.16,NaN
54,PO10055-N,MAT-1019,Spot,NaN,175.0
65,PO10066-P,MAT-1006,Contract,504.78,NaN
73,PO10074-N,MAT-1009,Spot,713.60,NaN
122,PO10123-N,MAT-1005,Spot,192.27,NaN
131,PO10132-S,MAT-1012,Contract,NaN,3.0
143,PO10144-N,MAT-1014,Contract,541.44,NaN
161,PO10162-S,MAT-1003,Spot,NaN,83.0


### 3.5 Números de lote repetidos

In [15]:
lote_repetido = compras["BatchNumber"].duplicated(keep=False)
compras[lote_repetido].sort_values("BatchNumber")[["POSampleID", "MaterialCode", "Center", "BatchNumber"]]

,POSampleID,MaterialCode,Center,BatchNumber
107,PO10108-S,MAT-1004,S,B1206
160,PO10161-S,MAT-1016,S,B1206
26,PO10027-S,MAT-1021,S,B5070
28,PO10029-N,MAT-1017,N,B5070
86,PO10087-P,MAT-1023,P,B8144
97,PO10098-P,MAT-1003,P,B8144


Tres lotes se repiten en órdenes de materiales distintos. El lote **no es un identificador único**.
**Decisión: no usarlo para cruzar datos.** No afecta el análisis.

## 4. Revisión de los archivos de referencia

### 4.1 Materiales

In [16]:
materiales.isna().sum()

MaterialCode                0
MaterialName                0
Category                    0
MaterialGroupName           0
MaterialGroupName2          0
MaterialType                0
IsUnknownGroup              0
ShelfLifeDays               7
StorageCostPerUnitPerDay    7
dtype: int64

In [17]:
# ¿Qué categorías no tienen vida útil ni costo de almacenamiento?
materiales[materiales["ShelfLifeDays"].isna()][["MaterialCode", "MaterialName", "Category"]]

,MaterialCode,MaterialName,Category
18,MAT-1019,Inbound Freight - Truck,Logistics Services
19,MAT-1020,Inbound Freight - Sea,Logistics Services
20,MAT-1021,Bonded Storage Fee,Logistics Services
21,MAT-1022,Palletizing Service,Logistics Services
22,MAT-1023,Misc Consumable 1,UNKNOWN
23,MAT-1024,Misc Consumable 2,UNKNOWN
24,MAT-1025,Misc Consumable 3,UNKNOWN


Los servicios logísticos (flete, almacenaje) y los materiales de grupo `UNKNOWN` no tienen vida útil
ni costo de almacenamiento. Es coherente con lo que dice el contexto.

### 4.2 Benchmark de precios

In [18]:
print("Combinaciones material-centro duplicadas:", benchmark.duplicated(["MaterialCode", "Center"]).sum())
print("Fechas del benchmark:", benchmark["PreviousPriceDate"].min(), "a", benchmark["PlannedPriceDate3"].max())
benchmark.head()

Combinaciones material-centro duplicadas: 0
Fechas del benchmark: 2024-01-02 a 2024-06-18


,MaterialCode,Center,PreviousPriceUSD,PreviousPriceDate,PlannedPrice2USD,PlannedPriceDate2,PlannedPrice3USD,PlannedPriceDate3
0,MAT-1001,S,31.68,2024-02-10,33.21,2024-03-17,34.28,2024-04-27
1,MAT-1001,P,32.77,2024-02-01,32.92,2024-03-23,34.05,2024-05-09
2,MAT-1002,N,19.76,2024-02-20,19.35,2024-03-26,19.41,2024-05-17
3,MAT-1002,S,18.81,2024-01-22,20.27,2024-03-15,19.93,2024-04-28
4,MAT-1002,P,20.05,2024-02-18,20.38,2024-03-29,21.93,2024-05-02


Observaciones:
- Las fechas del benchmark llegan hasta junio de 2024, pero hay órdenes hasta septiembre.
- Las columnas se llaman `PreviousPrice`, `PlannedPrice2` y `PlannedPrice3` (no existe un "PlannedPrice1").

**Decisión:** usar como precio de referencia el **promedio de los tres precios**. Es una simplificación;
el benchmark solo sirve para comprobar si el precio Spot fue realmente bajo, no para el cálculo principal.

In [19]:
benchmark["precio_referencia"] = benchmark[["PreviousPriceUSD", "PlannedPrice2USD", "PlannedPrice3USD"]].mean(axis=1).round(2)

### 4.3 Capacidad por centro

In [20]:
capacidad

,Center,CenterName,StorageCapacityUnits
0,N,North Plant,60000
1,S,South Plant,42000
2,P,Port Bonded Warehouse,18000


## 5. Unir las tablas
Uno todo a la tabla de compras con `merge(how="left")` para no perder ninguna orden.

In [21]:
datos = compras.merge(materiales, on="MaterialCode", how="left")
datos = datos.merge(capacidad, on="Center", how="left")
datos = datos.merge(benchmark[["MaterialCode", "Center", "precio_referencia"]],
                    on=["MaterialCode", "Center"], how="left")

print("Filas antes:", len(compras), "| filas después:", len(datos))

Filas antes: 165 | filas después: 165


## 6. Revisiones que necesitan la tabla unida

### 6.1 Órdenes más grandes que la capacidad total del centro

In [22]:
datos["flag_supera_capacidad"] = datos["OrderQuantity"] > datos["StorageCapacityUnits"]
datos[datos["flag_supera_capacidad"]][["POSampleID", "MaterialName", "Center", "OrderQuantity",
                                       "StorageCapacityUnits", "BuyingMode"]]

,POSampleID,MaterialName,Center,OrderQuantity,StorageCapacityUnits,BuyingMode
71,PO10072-P,Corrugated Cartons,P,19043,18000,Spot
138,PO10139-S,Amino Acid Blend,S,43496,42000,Spot
155,PO10156-N,Crude Palm Oil,N,60785,60000,Spot


Tres órdenes Spot son, por sí solas, **más grandes que toda la bodega del centro**. Es físicamente imposible
(probablemente un error de unidad o de digitación). Además son 20 a 50 veces más grandes que una orden normal,
así que dominarían cualquier suma. **Decisión: excluirlas del análisis principal** (en el análisis muestro
qué pasaría si se incluyeran).

### 6.2 Servicios y materiales sin grupo con días de almacenamiento

In [23]:
datos["es_servicio"] = datos["Category"] == "Logistics Services"
datos["es_desconocido"] = datos["Category"] == "UNKNOWN"

pd.crosstab(datos["Category"], datos["DaysInStorage"].notna())

DaysInStorage,False,True
Category,,
Logistics Services,0,22
MRO / Spare Parts,1,40
Packaging,1,25
Raw Materials,3,55
UNKNOWN,0,18


Los servicios (flete, almacenaje, paletizado) tienen `DaysInStorage` aunque no son algo físico que se guarde.
**Decisiones:**
- **Servicios:** días de almacenamiento = 0 (no ocupan bodega). Se comparan solo con precio + transporte.
- **UNKNOWN:** no tienen costo de almacenamiento ni vida útil, así que no se puede calcular su costo completo.
  **Se excluyen del análisis principal.**

In [24]:
datos["dias_almacenamiento"] = np.where(datos["es_servicio"], 0, datos["DaysInStorage"])

### 6.3 "Bonded Storage Fee" en centros que no son bodega aduanera

In [25]:
bonded_raro = (datos["MaterialCode"] == "MAT-1021") & (datos["Center"] != "P")
datos[bonded_raro][["POSampleID", "MaterialName", "Center", "BuyingMode"]]

,POSampleID,MaterialName,Center,BuyingMode
18,PO10019-N,Bonded Storage Fee,N,Spot
26,PO10027-S,Bonded Storage Fee,S,Spot
34,PO10035-N,Bonded Storage Fee,N,Contract
64,PO10065-N,Bonded Storage Fee,N,Spot
67,PO10068-N,Bonded Storage Fee,N,Contract
91,PO10092-S,Bonded Storage Fee,S,Spot
94,PO10095-N,Bonded Storage Fee,N,Contract
136,PO10137-N,Bonded Storage Fee,N,Spot


El cargo por bodega aduanera (bonded) aparece en las plantas Norte y Sur, que no son bodegas aduaneras.
**Decisión: lo dejo, pero queda como pregunta para el negocio.**

### 6.4 Lotes que se quedaron en bodega más tiempo que su vida útil

In [26]:
datos["supera_vida_util"] = datos["DaysInStorage"] > datos["ShelfLifeDays"]
pd.crosstab(datos["Category"], [datos["BuyingMode"], datos["supera_vida_util"]])

BuyingMode         Contract       Spot     
supera_vida_util      False True False True
Category                                   
Logistics Services       10    0    12    0
MRO / Spare Parts        31    0    10    0
Packaging                10    0    16    0
Raw Materials            36    1     4   17
UNKNOWN                  11    0     7    0

Casi todos los casos son **materias primas compradas en Spot**. No es un error de datos, es parte de lo que
hay que analizar, así que solo se marca.

### 6.5 Combinaciones material-centro sin benchmark

In [27]:
print("Órdenes sin precio de referencia:", datos["precio_referencia"].isna().sum(), "de", len(datos))

Órdenes sin precio de referencia: 28 de 165


## 7. Columna final: ¿qué filas entran al análisis?

In [28]:
datos["usar_en_analisis"] = (
    ~datos["flag_fuera_de_fecha"]
    & ~datos["flag_supera_capacidad"]
    & ~datos["flag_sin_transporte"]
    & ~datos["flag_sin_dias"]
    & ~datos["es_desconocido"]
)

print("Filas que entran al análisis:", datos["usar_en_analisis"].sum(), "de", len(datos))
pd.crosstab(datos["Category"], [datos["BuyingMode"], datos["usar_en_analisis"]])

Filas que entran al análisis: 134 de 165


BuyingMode         Contract       Spot     
usar_en_analisis      False True False True
Category                                   
Logistics Services        0   10     1   11
MRO / Spare Parts         1   30     0   10
Packaging                 2    8     2   14
Raw Materials             3   34     4   17
UNKNOWN                  11    0     7    0

## 8. Log de calidad de datos
Resumen de cada problema, cuántas filas afecta y qué decidí.

In [29]:
log = pd.DataFrame([
    ["Código de material con cero de más (MAT-0xxxx)", datos["flag_codigo_corregido"].sum(), "Corregido"],
    ["Centro distinto al sufijo del POSampleID", datos["flag_centro_inconsistente"].sum(), "Se deja Center, marcado"],
    ["Fecha fuera de ene-sep 2024", datos["flag_fuera_de_fecha"].sum(), "Excluido"],
    ["Sin costo de transporte", datos["flag_sin_transporte"].sum(), "Excluido del cálculo de costo"],
    ["Sin días de almacenamiento", datos["flag_sin_dias"].sum(), "Excluido del cálculo de costo"],
    ["Sin NegotiatorID", (datos["NegotiatorID"] == "SIN_DATO").sum(), "Reemplazado por SIN_DATO (no se usa)"],
    ["Cantidad mayor a la capacidad total del centro", datos["flag_supera_capacidad"].sum(), "Excluido (se prueba en sensibilidad)"],
    ["Número de lote repetido", lote_repetido.sum(), "No se usa como llave"],
    ["Servicio con días de almacenamiento", (datos["es_servicio"] & datos["DaysInStorage"].notna()).sum(), "Días = 0"],
    ["Material UNKNOWN (sin costo de almacenamiento)", datos["es_desconocido"].sum(), "Excluido"],
    ["Bonded Storage Fee en centro no aduanero", bonded_raro.sum(), "Se deja, pregunta al negocio"],
    ["Sin precio de referencia (benchmark)", datos["precio_referencia"].isna().sum(), "Solo afecta la comparación con benchmark"],
    ["Lote superó su vida útil", datos["supera_vida_util"].sum(), "Se deja: es parte del análisis"],
], columns=["problema", "filas_afectadas", "decision"])

log

,problema,filas_afectadas,decision
0,Código de material con cero de más (MAT-0xxxx),16,Corregido
1,Centro distinto al sufijo del POSampleID,2,"Se deja Center, marcado"
2,Fecha fuera de ene-sep 2024,1,Excluido
3,Sin costo de transporte,4,Excluido del cálculo de costo
4,Sin días de almacenamiento,5,Excluido del cálculo de costo
5,Sin NegotiatorID,12,Reemplazado por SIN_DATO (no se usa)
6,Cantidad mayor a la capacidad total del centro,3,Excluido (se prueba en sensibilidad)
7,Número de lote repetido,6,No se usa como llave
8,Servicio con días de almacenamiento,22,Días = 0
9,Material UNKNOWN (sin costo de almacenamiento),18,Excluido


## 9. Exportar los archivos limpios

In [30]:
columnas_finales = [
    "POSampleID", "SupplierCode", "MaterialCode", "MaterialName", "Category", "MaterialGroupName",
    "Center", "CenterName", "OrderDate", "BuyingMode", "NegotiatorID", "BatchNumber",
    "OrderQuantity", "UnitPriceUSD", "TransportCostUSD", "DaysInStorage", "dias_almacenamiento",
    "ShelfLifeDays", "StorageCostPerUnitPerDay", "StorageCapacityUnits", "precio_referencia",
    "es_servicio", "es_desconocido", "supera_vida_util",
    "flag_codigo_corregido", "flag_centro_inconsistente", "flag_fuera_de_fecha", "flag_sin_transporte",
    "flag_sin_dias", "flag_supera_capacidad", "usar_en_analisis",
]

datos[columnas_finales].to_excel("datos_limpios/compras_limpias.xlsx", index=False)
capacidad.to_excel("datos_limpios/capacidad_centros.xlsx", index=False)
log.to_excel("datos_limpios/log_calidad_datos.xlsx", index=False)

print("Archivos generados en la carpeta datos_limpios/")
print(os.listdir("datos_limpios"))

Archivos generados en la carpeta datos_limpios/
['capacidad_centros.xlsx', 'compras_limpias.xlsx', 'log_calidad_datos.xlsx']
